# Aula 7 — Extração de UCEs com PHYLUCE

A aula começa lendo o banco de matches produzido na Aula 6 e termina com os loci UCE
em um FASTA monolítico.

## 0. Preparar o runtime

O Google Drive guarda os **dados** entre as aulas, mas o runtime do Colab é temporário.
Programas instalados no runtime podem desaparecer quando a sessão termina.

Por isso, quando uma aula precisar de ferramentas externas, começaremos verificando se
o Conda já existe. Se não existir, ele será instalado antes de qualquer outra configuração.

> Esta deve ser a primeira célula executável do notebook, porque a instalação do Conda
> pode reiniciar o runtime.

In [ ]:
import shutil

if shutil.which("conda"):
    print("Conda já está disponível neste runtime.")
else:
    !pip install -q condacolab
    import condacolab
    condacolab.install()

### Verificar o Conda e configurar Bioconda

Usaremos a configuração recomendada pelo Bioconda: `conda-forge` com maior prioridade,
seguido de `bioconda`, e prioridade estrita.

Como `conda config --add` adiciona canais do menor para o maior nível de prioridade,
executamos primeiro `bioconda` e depois `conda-forge`.

In [ ]:
!conda --version
!conda config --remove-key channels 2>/dev/null || true
!conda config --add channels bioconda
!conda config --add channels conda-forge
!conda config --set channel_priority strict
!conda config --show channels

## 1. Retomar o projeto no Google Drive

Todas as práticas usam a mesma raiz:

`/content/drive/MyDrive/Bioinformatica_Biologia_Molecular`

Os resultados de uma aula são lidos pela aula seguinte. Assim, os **dados persistem**
mesmo quando o runtime do Colab é encerrado.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

ROOT = Path("/content/drive/MyDrive/Bioinformatica_Biologia_Molecular")
RUN = "SRR15736591"
SAMPLE = "hypochilus_petrunkevitchi_SRR15736591"

PASTAS = {
    "01_bancos": ROOT / "01_bancos",
    "02_blast": ROOT / "02_blast",
    "03_raw": ROOT / "03_sra_fastq" / "raw-fastq",
    "04_qc": ROOT / "04_qc_trimming",
    "04_trimmed": ROOT / "04_qc_trimming" / "trimmed",
    "05_assemblies": ROOT / "05_spades" / "spades-assemblies",
    "05_contigs": ROOT / "05_spades" / "spades-assemblies" / "contigs",
    "06_match": ROOT / "06_uce_match",
    "06_probes": ROOT / "06_uce_match" / "probes",
    "06_results": ROOT / "06_uce_match" / "uce-search-results",
    "07_taxon_sets": ROOT / "07_uce_extract" / "taxon-sets" / "all",
    "08_integracao": ROOT / "08_integracao",
    "ambientes": ROOT / "ambientes",
}

for pasta in PASTAS.values():
    pasta.mkdir(parents=True, exist_ok=True)

os.chdir(ROOT)

print("Diretório atual:", Path.cwd())
print("\nEstrutura principal do projeto:")
for chave, pasta in PASTAS.items():
    print(f"{chave:15s} -> {pasta.relative_to(ROOT)}")

## 2. Verificar as entradas da Aula 6

In [ ]:
PHY_ENV = "phyluce-1.7.3"
PHY_YML = PASTAS["ambientes"] / "phyluce-1.7.3-py36-Linux-conda.yml"

CONTIGS_DIR = PASTAS["05_contigs"]
MATCH_ROOT = PASTAS["06_match"]
RESULTS = PASTAS["06_results"]
DB = RESULTS / "probe.matches.sqlite"
TAXON_CONF = MATCH_ROOT / "taxon-set.conf"
TAXON_SET_DIR = PASTAS["07_taxon_sets"]

for arquivo in [DB, TAXON_CONF]:
    print(arquivo, "->", "OK" if arquivo.exists() else "AUSENTE")

if not DB.exists() or not TAXON_CONF.exists():
    raise FileNotFoundError("Resultados da Aula 6 não encontrados.")

## 3. Recriar/verificar o ambiente PHYLUCE

In [ ]:
envs = !conda env list
if not any(line.split() and line.split()[0] == PHY_ENV for line in envs if not line.startswith("#")):
    if not PHY_YML.exists():
        raise FileNotFoundError("YAML do PHYLUCE não encontrado. Execute primeiro a Aula 6.")
    !conda env create -n "$PHY_ENV" --file "$PHY_YML"
else:
    print("Ambiente PHYLUCE disponível.")

## 4. Criar a configuração dos loci presentes

In [ ]:
MATCH_CONF = TAXON_SET_DIR / "all-taxa-incomplete.conf"

!conda run -n "$PHY_ENV" phyluce_assembly_get_match_counts   --locus-db "$DB"   --taxon-list-config "$TAXON_CONF"   --taxon-group all   --incomplete-matrix   --output "$MATCH_CONF"

print(MATCH_CONF.read_text()[:3000])

## 5. Extrair as sequências UCE

In [ ]:
OUT_FASTA = TAXON_SET_DIR / "all-taxa-incomplete.fasta"
MISSING = TAXON_SET_DIR / "all-taxa-incomplete.incomplete"
LOGDIR = TAXON_SET_DIR / "log"
LOGDIR.mkdir(parents=True, exist_ok=True)

!conda run -n "$PHY_ENV" phyluce_assembly_get_fastas_from_match_counts   --contigs "$CONTIGS_DIR"   --locus-db "$DB"   --match-count-output "$MATCH_CONF"   --output "$OUT_FASTA"   --incomplete-matrix "$MISSING"   --log-path "$LOGDIR"

## 6. Visualizar e contar os loci

In [ ]:
!head -20 "$OUT_FASTA"

In [ ]:
def read_fasta(path):
    records = []
    header = None
    seq = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if header is not None:
                    records.append((header, "".join(seq)))
                header = line[1:]
                seq = []
            else:
                seq.append(line)
        if header is not None:
            records.append((header, "".join(seq)))
    return records

records = read_fasta(OUT_FASTA)
lengths = [len(s) for _, s in records]

print("Registros FASTA:", len(records))
print("Mínimo:", min(lengths) if lengths else 0)
print("Máximo:", max(lengths) if lengths else 0)
print("Média:", round(sum(lengths)/len(lengths), 1) if lengths else 0)

for h, s in records[:10]:
    print(h, len(s))

## Saída para a próxima aula

A integração final usará:

`07_uce_extract/taxon-sets/all/all-taxa-incomplete.fasta`

Esse caminho segue a lógica utilizada no tutorial do PHYLUCE para organizar os conjuntos de táxons.